In [1]:
import numpy as np
import pandas as pd
from pyhive import presto
from datetime import timedelta,datetime
import json
import requests
import math
import os


pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings("ignore")
conn = presto.connect(
        host="presto-gateway.serving.data.plectrum.dev",
        port=80,
        username="manoj.ravirajan@rapido.bike"
    )
presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'
conn1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
conn2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
conn3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
conn4= presto.connect('processing.processing.data.production.internal',presto_port,username)

### Loading data and merging

In [2]:
daily_segment_df = pd.read_sql(f"""select * from hive.experiments_internal.customer_segmentation_amj2025""", conn2)

In [3]:
# rha_qc_segment_df = pd.read_sql(f"""select * from hive.experiments_internal.rha_qc_segment_20250406""", conn3)
rha_qc_segment_df = pd.read_sql(f"""select customer_id, 
                                           office_tag, os, 
                                           travelridehailing, 
                                           quick_commerce from hive.experiments_internal.rha_qc_usecase_segment_20250406""", conn2)

In [4]:
# rf_daily_df = pd.read_sql(f"""select * from hive.experiments_internal.rf_daily_30days_20250317""", conn2)
rf_daily_df = pd.read_sql(f"""select * from hive.experiments_internal.rf_daily_30days_20250317_v1""", conn2)

In [7]:
# olf_g2n_df = pd.read_sql(f"""select * from hive.experiments_internal.olf_g2n_30days_20250317""", conn3)
# olf_g2n_df = pd.read_sql(f"""select * from hive.experiments_internal.olf_g2n_30days_20250317_v1""", conn2)
olf_g2n_df = pd.read_sql(f"""select * from hive.experiments_internal.olf_g2n_30days_20250317_v2""", conn2)

In [8]:
olf_geo_usecase_df = pd.read_sql(f"""select * from hive.experiments_internal.olf_geo_usecase_20250317""", conn2)

In [9]:
retries_aoferr_df = pd.read_sql(f"""select * from hive.experiments_internal.retries_aoferr_30days_20250317""", conn2)

In [10]:
ccfr_comm_df = pd.read_sql(f"""select * from hive.experiments_internal.ratings_custcapcomm_30days_20250317_30days""", conn2)

In [11]:
sup_tikt_df =  pd.read_sql(f"""select * from hive.experiments_internal.support_ticket_20250317_30days""", conn2)
sup_tikt_df = sup_tikt_df[sup_tikt_df['customer_id']!='customer_id']

In [12]:
cust_rrsessns_df = pd.read_sql(f"""select * from hive.experiments_internal.customer_rr_sessions_20250317_30days""", conn2)

In [13]:
merged_df = daily_segment_df.merge(rha_qc_segment_df, on = 'customer_id', how='left')
merged_df = merged_df.merge(rf_daily_df, on = 'customer_id', how='left')
merged_df = merged_df.merge(olf_g2n_df, on = 'customer_id', how='left')
merged_df = merged_df.merge(olf_geo_usecase_df, on = 'customer_id', how='left')
merged_df = merged_df.merge(retries_aoferr_df, on = 'customer_id', how='left')
merged_df = merged_df.merge(ccfr_comm_df, on = 'customer_id', how='left')
merged_df = merged_df.merge(sup_tikt_df, on = 'customer_id', how='left')
merged_df = merged_df.merge(cust_rrsessns_df, on = 'customer_id', how='left')
merged_df.head()

,customer_id,segment,office_tag,os,travelridehailing,quick_commerce,ao_session,fe_session,rr_session,ao2fe_delta_mean,fe2rr_delta_mean,ao2rr_delta_mean,rr_count,net_count,cobrm_count,cobra_count,ocara_count,canceltn_count,quick_cobra,quick_ocara,cobra_ttc,ocara_ttc,tta,rider_canceltn_count,expired_count,aborted_count,haps_orders,bid_orders,negative_bid,positive_bid,gross_psm_gps,gross_psm_fav_recent_history,gross_psm_google_search,gross_psm_stickylocation,gross_psm_hotspot,net_psm_gps,net_psm_fav_recent_history,net_psm_google_search,net_psm_fav_stickylocation,net_psm_hotspot,office_gross,transit_gross,office_net,transit_net,total_aos,total_ao_days,total_ao_sessions,total_fes,total_fe_days,total_fe_sessions,total_rrs,total_rr_days,total_rr_sessions,gross_orders,feedback_rating,rating_5,rating_4,rating_3,rating_2,rating_1,customer_call_captain,captain_call_customer,customer_msg_captain,captain_msg_customer,call_communication,msg_communication,communication,ocara_no,ocara_with_comm,ocara_without_comm,ocara_with_call,ocara_without_call,support_ticket,customer_rr_sessions
0,5e29174052f0da0a79fb9926,6-QUARTERLY,office,Android,no,yes,4,3,1,119.33333333333333,105.0,105.0,3,1,0,1,1,2,0,0,57.0,250.0,594.5,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,1,0,8,3,4,7.0,2.0,4.0,3.0,1.0,1.0,3,0,0,0,0,0,0,0,2,0,1,2,1,2,1,1,0,1,0,NaN,1
1,61d72742dd82c2c08b2055fd,8-RR-INACTIVE,other,Android,yes,yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,67b348c94aa07b7847a59a90,6-QUARTERLY,other,Android,yes,yes,7,7,5,42.2,59.8,108.33333333333333,9,7,0,0,2,2,0,0,,1017.0,169.14285714285714,1,0,0,5,0,0,0,0,4,3,0,2,0,3,3,0,1,0,2,0,1,7,5,7,22.0,8.0,12.0,9.0,6.0,8.0,9,0,0,0,0,0,0,0,5,1,4,5,4,6,2,1,1,1,1,NaN,8
3,634a996109c7bf2528e04bc9,5-MONTHLY,other,iOS,blank,blank,1,1,0,26.0,,,1,1,0,0,0,0,0,0,,,27.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,4.0,1.0,2.0,1.0,1.0,1.0,1,0,0,0,0,0,0,0,1,0,1,1,1,1,0,0,0,0,0,NaN,1
4,64ddedbb43ffc9ce64b116ac,2-HH-inActive,office,Android,yes,yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
merged_df.columns

Index(['customer_id', 'segment', 'office_tag', 'os', 'travelridehailing',
       'quick_commerce', 'ao_session', 'fe_session', 'rr_session',
       'ao2fe_delta_mean', 'fe2rr_delta_mean', 'ao2rr_delta_mean', 'rr_count',
       'net_count', 'cobrm_count', 'cobra_count', 'ocara_count',
       'canceltn_count', 'quick_cobra', 'quick_ocara', 'cobra_ttc',
       'ocara_ttc', 'tta', 'rider_canceltn_count', 'expired_count',
       'aborted_count', 'haps_orders', 'bid_orders', 'negative_bid',
       'positive_bid', 'gross_psm_gps', 'gross_psm_fav_recent_history',
       'gross_psm_google_search', 'gross_psm_stickylocation',
       'gross_psm_hotspot', 'net_psm_gps', 'net_psm_fav_recent_history',
       'net_psm_google_search', 'net_psm_fav_stickylocation',
       'net_psm_hotspot', 'office_gross', 'transit_gross', 'office_net',
       'transit_net', 'total_aos', 'total_ao_days', 'total_ao_sessions',
       'total_fes', 'total_fe_days', 'total_fe_sessions', 'total_rrs',
       'total_rr_days', 

In [15]:
merged_df.to_parquet('merged_df_20250423.parquet')

#### Grouping

In [16]:
merged_df = pd.read_parquet('merged_df_20250423.parquet')

In [17]:
merged_df.shape

(4929725, 74)

In [18]:
merged_df

,customer_id,segment,office_tag,os,travelridehailing,quick_commerce,ao_session,fe_session,rr_session,ao2fe_delta_mean,fe2rr_delta_mean,ao2rr_delta_mean,rr_count,net_count,cobrm_count,cobra_count,ocara_count,canceltn_count,quick_cobra,quick_ocara,cobra_ttc,ocara_ttc,tta,rider_canceltn_count,expired_count,aborted_count,haps_orders,bid_orders,negative_bid,positive_bid,gross_psm_gps,gross_psm_fav_recent_history,gross_psm_google_search,gross_psm_stickylocation,gross_psm_hotspot,net_psm_gps,net_psm_fav_recent_history,net_psm_google_search,net_psm_fav_stickylocation,net_psm_hotspot,office_gross,transit_gross,office_net,transit_net,total_aos,total_ao_days,total_ao_sessions,total_fes,total_fe_days,total_fe_sessions,total_rrs,total_rr_days,total_rr_sessions,gross_orders,feedback_rating,rating_5,rating_4,rating_3,rating_2,rating_1,customer_call_captain,captain_call_customer,customer_msg_captain,captain_msg_customer,call_communication,msg_communication,communication,ocara_no,ocara_with_comm,ocara_without_comm,ocara_with_call,ocara_without_call,support_ticket,customer_rr_sessions
0,5e29174052f0da0a79fb9926,6-QUARTERLY,office,Android,no,yes,4,3,1,119.33333333333333,105.0,105.0,3,1,0,1,1,2,0,0,57.0,250.0,594.5,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,1,0,8,3,4,7.0,2.0,4.0,3.0,1.0,1.0,3,0,0,0,0,0,0,0,2,0,1,2,1,2,1,1,0,1,0,None,1
1,61d72742dd82c2c08b2055fd,8-RR-INACTIVE,other,Android,yes,yes,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,67b348c94aa07b7847a59a90,6-QUARTERLY,other,Android,yes,yes,7,7,5,42.2,59.8,108.33333333333333,9,7,0,0,2,2,0,0,,1017.0,169.14285714285714,1,0,0,5,0,0,0,0,4,3,0,2,0,3,3,0,1,0,2,0,1,7,5,7,22.0,8.0,12.0,9.0,6.0,8.0,9,0,0,0,0,0,0,0,5,1,4,5,4,6,2,1,1,1,1,None,8
3,634a996109c7bf2528e04bc9,5-MONTHLY,other,iOS,blank,blank,1,1,0,26.0,,,1,1,0,0,0,0,0,0,,,27.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,4.0,1.0,2.0,1.0,1.0,1.0,1,0,0,0,0,0,0,0,1,0,1,1,1,1,0,0,0,0,0,None,1
4,64ddedbb43ffc9ce64b116ac,2-HH-inActive,office,Android,yes,yes,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4929720,647d936f1edcd31fa9fe5e0a,6-QUARTERLY,office,Android,no,yes,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
4929721,67649fb866024a7fb199e66b,1-HH-Active,office,Android,yes,yes,5,4,1,11.75,39.0,54.0,2,1,0,1,0,1,0,0,51.0,,3.0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,1,0,0,0,0,5,4,5,6.0,3.0,4.0,2.0,1.0,1.0,2,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,None,1
4929722,645c5e6a5d0cb382822829ea,4-WEEKLY,other,Android,yes,yes,12,9,6,19.0,26.666666666666668,49.2,10,9,0,0,1,1,0,0,,664.0,40.111111111111114,1,0,0,9,0,0,0,2,0,0,7,0,2,0,0,6,0,0,2,0,2,12,10,12,24.0,10.0,13.0,10.0,9.0,10.0,10,3,3,0,0,0,0,0,5,6,4,5,7,9,1,0,1,0,1,1,10
4929723,632989490a5820e31fde3d88,6-QUARTERLY,other,Android,yes,yes,1,0,0,,,,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Non

In [19]:
merged_df['os'].value_counts(normalize=True)

os
Android    0.817882
iOS        0.141033
Blank      0.041084
Name: proportion, dtype: float64

In [20]:
merged_df = merged_df[~merged_df['os'].isin(['Blank'])]

In [21]:
merged_df.customer_id.nunique()

4727191

In [22]:
merged_df.rename(columns={'net_psm_fav_stickylocation':'net_psm_stickylocation'}, inplace=True)

In [23]:
merged_df.columns

Index(['customer_id', 'segment', 'office_tag', 'os', 'travelridehailing',
       'quick_commerce', 'ao_session', 'fe_session', 'rr_session',
       'ao2fe_delta_mean', 'fe2rr_delta_mean', 'ao2rr_delta_mean', 'rr_count',
       'net_count', 'cobrm_count', 'cobra_count', 'ocara_count',
       'canceltn_count', 'quick_cobra', 'quick_ocara', 'cobra_ttc',
       'ocara_ttc', 'tta', 'rider_canceltn_count', 'expired_count',
       'aborted_count', 'haps_orders', 'bid_orders', 'negative_bid',
       'positive_bid', 'gross_psm_gps', 'gross_psm_fav_recent_history',
       'gross_psm_google_search', 'gross_psm_stickylocation',
       'gross_psm_hotspot', 'net_psm_gps', 'net_psm_fav_recent_history',
       'net_psm_google_search', 'net_psm_stickylocation', 'net_psm_hotspot',
       'office_gross', 'transit_gross', 'office_net', 'transit_net',
       'total_aos', 'total_ao_days', 'total_ao_sessions', 'total_fes',
       'total_fe_days', 'total_fe_sessions', 'total_rrs', 'total_rr_days',
       'tot

In [24]:
cols_to_convert = [ 'rr_count', 'net_count', 'cobrm_count', 'cobra_count', 'ocara_count',
       'canceltn_count', 'quick_cobra', 'quick_ocara', 'cobra_ttc',
       'ocara_ttc', 'tta', 'rider_canceltn_count', 'expired_count', 'aborted_count', 'haps_orders',
       'bid_orders', 'negative_bid', 'positive_bid', 'gross_psm_gps',
       'gross_psm_fav_recent_history', 'gross_psm_google_search', 'gross_psm_stickylocation', 'gross_psm_hotspot',
       'net_psm_gps', 'net_psm_fav_recent_history', 'net_psm_google_search', 'net_psm_stickylocation', 'net_psm_hotspot',
       'ao_session', 'fe_session', 'rr_session', 'ao2fe_delta_mean', 'fe2rr_delta_mean', 'ao2rr_delta_mean',
        'office_gross', 'transit_gross', 'office_net', 'transit_net', 'total_aos',
       'total_ao_days', 'total_ao_sessions', 'total_fes', 'total_fe_days',
       'total_fe_sessions', 'total_rrs', 'total_rr_days', 'total_rr_sessions',
       'gross_orders', 'feedback_rating', 'rating_5', 'rating_4', 'rating_3',
       'rating_2', 'rating_1', 'customer_call_captain',
       'captain_call_customer', 'customer_msg_captain', 'captain_msg_customer',
       'call_communication', 'msg_communication', 'communication', 'ocara_no',
       'ocara_with_comm', 'ocara_without_comm', 'ocara_with_call',
       'ocara_without_call','support_ticket','customer_rr_sessions']

for col in cols_to_convert:
    merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')

In [25]:
merged_df['net_customers'] = merged_df['net_count'].apply(lambda x: 1 if x>0 else 0)
merged_df['gross_customers'] = merged_df['rr_count'].apply(lambda x: 1 if x>0 else 0)

In [26]:
# Revise Daily/Weekly/Monthy... threshold
# Retries - effort
    # total fe per total active fe days (ao/fe/rr)
    # total fe per total fe session
# gsr2net
# session length
# Use CT session data as proxy
# acceptance time
# Add p25 for all cancellation metric
# Rating distribution
# Captain cancellation
# Customer/Captain contact %
# Customer support contact ratio
# Rest
# Bad captain mapping %
# Office \ Transit \ Other
# Acquired at transit
# Ocara without contact and with contact
# Drop selection mode
# Median active days

In [27]:
merged_df['aos_per_session'] = round(merged_df['total_aos']/merged_df['total_ao_sessions'],2)
merged_df['aos_per_day'] = round(merged_df['total_aos']/merged_df['total_ao_days'],2)

merged_df['fes_per_session'] = round(merged_df['total_fes']/merged_df['total_fe_sessions'],2)
merged_df['fes_per_day'] = round(merged_df['total_fes']/merged_df['total_fe_days'],2)

merged_df['rrs_per_session'] = round(merged_df['total_rrs']/merged_df['total_rr_sessions'],2)
merged_df['rrs_per_day'] = round(merged_df['total_rrs']/merged_df['total_rr_days'],2)

In [28]:
merged_df.columns

Index(['customer_id', 'segment', 'office_tag', 'os', 'travelridehailing',
       'quick_commerce', 'ao_session', 'fe_session', 'rr_session',
       'ao2fe_delta_mean', 'fe2rr_delta_mean', 'ao2rr_delta_mean', 'rr_count',
       'net_count', 'cobrm_count', 'cobra_count', 'ocara_count',
       'canceltn_count', 'quick_cobra', 'quick_ocara', 'cobra_ttc',
       'ocara_ttc', 'tta', 'rider_canceltn_count', 'expired_count',
       'aborted_count', 'haps_orders', 'bid_orders', 'negative_bid',
       'positive_bid', 'gross_psm_gps', 'gross_psm_fav_recent_history',
       'gross_psm_google_search', 'gross_psm_stickylocation',
       'gross_psm_hotspot', 'net_psm_gps', 'net_psm_fav_recent_history',
       'net_psm_google_search', 'net_psm_stickylocation', 'net_psm_hotspot',
       'office_gross', 'transit_gross', 'office_net', 'transit_net',
       'total_aos', 'total_ao_days', 'total_ao_sessions', 'total_fes',
       'total_fe_days', 'total_fe_sessions', 'total_rrs', 'total_rr_days',
       'tot

In [29]:
def agg_func(col):
    
    grouper = col
    print(grouper)
    
    segment_agg_df = merged_df\
    .groupby(grouper)\
    .agg(
         gross_customers = ('gross_customers', 'sum'),
         net_customers = ('net_customers', 'sum'),
         
         ao_session = ('ao_session', 'sum'),
         fe_session = ('fe_session', 'sum'),
         rr_session = ('rr_session', 'sum'),

         rr_count = ('rr_count', 'sum'),
         net_count = ('net_count', 'sum'),
         cobrm_count = ('cobrm_count', 'sum'),
         cobra_count = ('cobra_count', 'sum'),
         ocara_count = ('ocara_count', 'sum'),
         expired_count = ('expired_count', 'sum'),
         aborted_count = ('aborted_count', 'sum'),
         canceltn_count = ('canceltn_count', 'sum'),
         rider_canceltn_count = ('rider_canceltn_count', 'sum'),
        
         quick_cobra = ('quick_cobra', 'sum'),
         quick_ocara = ('quick_ocara', 'sum'),
         
         cobra_ttc_mean = ('cobra_ttc', 'mean'),
         cobra_ttc_p25=('cobra_ttc', lambda x: x.quantile(0.25)),
         cobra_ttc_p50 = ('cobra_ttc', 'median'),
         cobra_ttc_p75=('cobra_ttc', lambda x: x.quantile(0.75)),
         cobra_ttc_p90=('cobra_ttc', lambda x: x.quantile(0.90)),
         cobra_ttc_p95=('cobra_ttc', lambda x: x.quantile(0.95)),
    
         ocara_ttc_mean = ('ocara_ttc', 'mean'),
         ocara_ttc_p25=('ocara_ttc', lambda x: x.quantile(0.25)),
         ocara_ttc_p50 = ('ocara_ttc', 'median'),
         ocara_ttc_p75=('ocara_ttc', lambda x: x.quantile(0.75)),
         ocara_ttc_p90=('ocara_ttc', lambda x: x.quantile(0.90)),
         ocara_ttc_p95=('ocara_ttc', lambda x: x.quantile(0.95)),

         tta_mean = ('tta', 'mean'),
         tta_p25=('tta', lambda x: x.quantile(0.25)),
         tta_p50 = ('tta', 'median'),
         tta_p75=('tta', lambda x: x.quantile(0.75)),
         tta_p90=('tta', lambda x: x.quantile(0.90)),
         tta_p95=('tta', lambda x: x.quantile(0.95)),
    
         haps_orders = ('haps_orders', 'sum'),
         bid_orders = ('bid_orders', 'sum'),
         negative_bid = ('negative_bid', 'sum'),
         positive_bid = ('positive_bid', 'sum'),
    
         gross_psm_gps = ('gross_psm_gps', 'sum'),
         gross_psm_fav_recent_history = ('gross_psm_fav_recent_history', 'sum'),
         gross_psm_google_search = ('gross_psm_google_search', 'sum'),
         gross_psm_stickylocation = ('gross_psm_stickylocation', 'sum'),
         gross_psm_hotspot = ('gross_psm_hotspot', 'sum'),
        
         net_psm_gps = ('net_psm_gps', 'sum'),
         net_psm_fav_recent_history = ('net_psm_fav_recent_history', 'sum'),
         net_psm_google_search = ('net_psm_google_search', 'sum'),
         net_psm_stickylocation = ('net_psm_stickylocation', 'sum'),
         net_psm_hotspot = ('net_psm_hotspot', 'sum'),
         
         office_gross = ('office_gross', 'sum'),
         transit_gross = ('transit_gross', 'sum'),
         office_net = ('office_net', 'sum'),
         transit_net = ('transit_net', 'sum'),

         ao2fe_delta_mean = ('ao2fe_delta_mean', 'mean'),
         ao2fe_delta_p25=('ao2fe_delta_mean', lambda x: x.quantile(0.25)),
         ao2fe_delta_p50 = ('ao2fe_delta_mean', 'median'),
         ao2fe_delta_p75=('ao2fe_delta_mean', lambda x: x.quantile(0.75)),
         ao2fe_delta_p90=('ao2fe_delta_mean', lambda x: x.quantile(0.90)),
         ao2fe_delta_p95=('ao2fe_delta_mean', lambda x: x.quantile(0.95)),

         fe2rr_delta_mean = ('fe2rr_delta_mean', 'mean'),
         fe2rr_delta_p25=('fe2rr_delta_mean', lambda x: x.quantile(0.25)),
         fe2rr_delta_p50 = ('fe2rr_delta_mean', 'median'),
         fe2rr_delta_p75=('fe2rr_delta_mean', lambda x: x.quantile(0.75)),
         fe2rr_delta_p90=('fe2rr_delta_mean', lambda x: x.quantile(0.90)),
         fe2rr_delta_p95=('fe2rr_delta_mean', lambda x: x.quantile(0.95)),

         ao2rr_delta_mean = ('ao2rr_delta_mean', 'mean'),
         ao2rr_delta_p25=('ao2rr_delta_mean', lambda x: x.quantile(0.25)),
         ao2rr_delta_p50 = ('ao2rr_delta_mean', 'median'),
         ao2rr_delta_p75=('ao2rr_delta_mean', lambda x: x.quantile(0.75)),
         ao2rr_delta_p90=('ao2rr_delta_mean', lambda x: x.quantile(0.90)),
         ao2rr_delta_p95=('ao2rr_delta_mean', lambda x: x.quantile(0.95)),

         aos_per_session_mean = ('aos_per_session', 'mean'),
         aos_per_session_p25=('aos_per_session', lambda x: x.quantile(0.25)),
         aos_per_session_p50 = ('aos_per_session', 'median'),
         aos_per_session_p75=('aos_per_session', lambda x: x.quantile(0.75)),
         aos_per_session_p90=('aos_per_session', lambda x: x.quantile(0.90)),
         aos_per_session_p95=('aos_per_session', lambda x: x.quantile(0.95)),

         aos_per_day_mean = ('aos_per_day', 'mean'),
         aos_per_day_p25=('aos_per_day', lambda x: x.quantile(0.25)),
         aos_per_day_p50 = ('aos_per_day', 'median'),
         aos_per_day_p75=('aos_per_day', lambda x: x.quantile(0.75)),
         aos_per_day_p90=('aos_per_day', lambda x: x.quantile(0.90)),
         aos_per_day_p95=('aos_per_day', lambda x: x.quantile(0.95)),

         fes_per_session_mean = ('fes_per_session', 'mean'),
         fes_per_session_p25=('fes_per_session', lambda x: x.quantile(0.25)),
         fes_per_session_p50 = ('fes_per_session', 'median'),
         fes_per_session_p75=('fes_per_session', lambda x: x.quantile(0.75)),
         fes_per_session_p90=('fes_per_session', lambda x: x.quantile(0.90)),
         fes_per_session_p95=('fes_per_session', lambda x: x.quantile(0.95)),

         fes_per_day_mean = ('fes_per_day', 'mean'),
         fes_per_day_p25=('fes_per_day', lambda x: x.quantile(0.25)),
         fes_per_day_p50 = ('fes_per_day', 'median'),
         fes_per_day_p75=('fes_per_day', lambda x: x.quantile(0.75)),
         fes_per_day_p90=('fes_per_day', lambda x: x.quantile(0.90)),
         fes_per_day_p95=('fes_per_day', lambda x: x.quantile(0.95)),

         rrs_per_session_mean = ('rrs_per_session', 'mean'),
         rrs_per_session_p25=('rrs_per_session', lambda x: x.quantile(0.25)),
         rrs_per_session_p50 = ('rrs_per_session', 'median'),
         rrs_per_session_p75=('rrs_per_session', lambda x: x.quantile(0.75)),
         rrs_per_session_p90=('rrs_per_session', lambda x: x.quantile(0.90)),
         rrs_per_session_p95=('rrs_per_session', lambda x: x.quantile(0.95)),

         rrs_per_day_mean = ('rrs_per_day', 'mean'),
         rrs_per_day_p25=('rrs_per_day', lambda x: x.quantile(0.25)),
         rrs_per_day_p50 = ('rrs_per_day', 'median'),
         rrs_per_day_p75=('rrs_per_day', lambda x: x.quantile(0.75)),
         rrs_per_day_p90=('rrs_per_day', lambda x: x.quantile(0.90)),
         rrs_per_day_p95=('rrs_per_day', lambda x: x.quantile(0.95)),

         feedback_rating = ('feedback_rating', 'sum'),
         rating_5 = ('rating_5', 'sum'),
         rating_4 = ('rating_4', 'sum'),
         rating_3 = ('rating_3', 'sum'),
         rating_2 = ('rating_2', 'sum'),
         rating_1 = ('rating_1', 'sum'),
         customer_call_captain = ('customer_call_captain', 'sum'),
         captain_call_customer = ('captain_call_customer', 'sum'), 
         customer_msg_captain = ('customer_msg_captain', 'sum'),
         captain_msg_customer = ('captain_msg_customer', 'sum'),
         call_communication = ('call_communication', 'sum'),
         msg_communication = ('msg_communication', 'sum'),
         communication = ('communication', 'sum'),
         ocara_no = ('ocara_no', 'sum'),
         ocara_with_comm = ('ocara_with_comm', 'sum'),
         ocara_without_comm = ('ocara_without_comm', 'sum'),
         ocara_with_call = ('ocara_with_call', 'sum'),
         ocara_without_call = ('ocara_without_call', 'sum'),
         support_ticket = ('support_ticket', 'sum'),
         customer_rr_sessions = ('customer_rr_sessions', 'sum')
        
        )\
    .reset_index()\
    .sort_values(by=grouper, ascending=True)

    segment_agg_df['gsr2net'] = round(segment_agg_df['net_count']*100/segment_agg_df['customer_rr_sessions'],2)
    segment_agg_df['rpc'] = round(segment_agg_df['net_count']/segment_agg_df['net_customers'],2)

    segment_agg_df['a02fe'] = round(segment_agg_df['fe_session']*100/segment_agg_df['ao_session'],2)
    segment_agg_df['fe2rr'] = round(segment_agg_df['rr_session']*100/segment_agg_df['fe_session'],2)
    segment_agg_df['ao2rr'] = round(segment_agg_df['rr_session']*100/segment_agg_df['ao_session'],2)
    
    segment_agg_df['g2n'] = round(segment_agg_df['net_count']*100/segment_agg_df['rr_count'],2)
    segment_agg_df['haps%'] = round(segment_agg_df['haps_orders']*100/segment_agg_df['net_count'],2)

    # cobrm/cobra/ocara/quick_cobra/quick_ocara/(expired_count+aborted_count) % = (cobrm/cobra/ocara/quick_cobra/quick_ocara/(expired_count+aborted_count))/rr
    for col in ['cobrm_count','cobra_count','ocara_count']:
        segment_agg_df[col.split('_')[0]+'%'] = round(segment_agg_df[col]*100/segment_agg_df['rr_count'],2)
    
    for col in ['quick_cobra','quick_ocara']:
        segment_agg_df[col +'%'] = round(segment_agg_df[col]*100/segment_agg_df['rr_count'],2)
    
    segment_agg_df['expired%'] = round((segment_agg_df['expired_count'] + segment_agg_df['aborted_count'])*100/segment_agg_df['rr_count'],2)

    segment_agg_df['positive_bid%'] = round(segment_agg_df['positive_bid']*100/segment_agg_df['bid_orders'],2)
    segment_agg_df['negative_bid%'] = round(segment_agg_df['negative_bid']*100/segment_agg_df['bid_orders'],2)

    # 'gross_psm_gps','gross_psm_fav_recent_history', 'gross_psm_google_search', 'gross_psm_stickylocation', 'gross_psm_hotspot'
    for col in ['gross_psm_gps','gross_psm_fav_recent_history', 'gross_psm_google_search', 'gross_psm_stickylocation', 'gross_psm_hotspot']:
        segment_agg_df[col +'%'] = round(segment_agg_df[col]*100/segment_agg_df['rr_count'],2)
    # 'net_psm_gps','net_psm_fav_recent_history', 'net_psm_google_search', 'net_psm_fav_stickylocation', 'net_psm_hotspot'
    for col in ['net_psm_gps','net_psm_fav_recent_history', 'net_psm_google_search', 'net_psm_stickylocation', 'net_psm_hotspot']:
        segment_agg_df[col +'%'] = round(segment_agg_df[col]*100/segment_agg_df['net_count'],2)

    for col in ['office_gross','transit_gross']:
        segment_agg_df[col +'%'] = round(segment_agg_df[col]*100/segment_agg_df['rr_count'],2)

    for col in ['office_net','transit_net']:
        segment_agg_df[col +'%'] = round(segment_agg_df[col]*100/segment_agg_df['net_count'],2)

    segment_agg_df['% Bid'] = round(segment_agg_df['bid_orders']*100/segment_agg_df['rr_count'],2)
    
    segment_agg_df['% Gross customers'] = round(segment_agg_df['gross_customers']*100/ segment_agg_df.groupby('segment')['gross_customers'].transform('sum'),2)
    segment_agg_df['% Gross Orders'] = round(segment_agg_df['rr_count']*100/ segment_agg_df.groupby('segment')['rr_count'].transform('sum'),2)
    
    segment_agg_df['% rider_canceltn'] = round(segment_agg_df['rider_canceltn_count']*100/segment_agg_df['rr_count'],2)

    segment_agg_df['% rated_orders'] = round(segment_agg_df['feedback_rating']*100/segment_agg_df['net_count'],2)

    segment_agg_df['% rating_5'] = round(segment_agg_df['rating_5']*100/segment_agg_df['feedback_rating'],2)
    segment_agg_df['% rating_4'] = round(segment_agg_df['rating_4']*100/segment_agg_df['feedback_rating'],2)
    segment_agg_df['% rating_3'] = round(segment_agg_df['rating_3']*100/segment_agg_df['feedback_rating'],2)
    segment_agg_df['% rating_2'] = round(segment_agg_df['rating_2']*100/segment_agg_df['feedback_rating'],2)
    segment_agg_df['% rating_1'] = round(segment_agg_df['rating_1']*100/segment_agg_df['feedback_rating'],2)

    segment_agg_df['average_rating'] =  round(((segment_agg_df['% rating_5']*5) + (segment_agg_df['% rating_4']*4) + (segment_agg_df['% rating_3']*3) + (segment_agg_df['% rating_2']*2) + (segment_agg_df['% rating_1']*1))/100,2)

    segment_agg_df['% call_communication'] = round(segment_agg_df['call_communication']*100/segment_agg_df['rr_count'],2)
    segment_agg_df['% msg_communication'] = round(segment_agg_df['msg_communication']*100/segment_agg_df['rr_count'],2)
    segment_agg_df['% communication'] = round(segment_agg_df['communication']*100/segment_agg_df['rr_count'],2)

    segment_agg_df['% ocara_with_comm'] = round(segment_agg_df['ocara_with_comm']*100/segment_agg_df['ocara_no'],2)
    segment_agg_df['% ocara_without_comm'] = round(segment_agg_df['ocara_without_comm']*100/segment_agg_df['ocara_no'],2)
    segment_agg_df['% ocara_with_call'] = round(segment_agg_df['ocara_with_call']*100/segment_agg_df['ocara_no'],2)
    segment_agg_df['% ocara_without_call'] = round(segment_agg_df['ocara_without_call']*100/segment_agg_df['ocara_no'],2)

    segment_agg_df['% support_ticket'] = round(segment_agg_df['support_ticket']*100/segment_agg_df['net_count'],2)

    cols1 = ['gross_customers', 'net_customers', 'ao_session', 'fe_session', 'rr_session', 'customer_rr_sessions', 'rr_count', 'net_count', 'cobrm_count', 'cobra_count', 'ocara_count', 'expired_count', 'aborted_count', 'canceltn_count', 'quick_cobra', 'quick_ocara', 'cobra_ttc_mean', 'cobra_ttc_p25', 'cobra_ttc_p50', 'cobra_ttc_p75', 'cobra_ttc_p90', 'cobra_ttc_p95', 'ocara_ttc_mean', 'ocara_ttc_p25', 'ocara_ttc_p50', 'ocara_ttc_p75', 'ocara_ttc_p90', 'ocara_ttc_p95', 'haps_orders', 'bid_orders', 'negative_bid', 'positive_bid', 'gross_psm_gps', 'gross_psm_fav_recent_history', 'gross_psm_google_search', 'gross_psm_stickylocation', 'gross_psm_hotspot', 'net_psm_gps', 'net_psm_fav_recent_history', 'net_psm_google_search', 'net_psm_stickylocation', 'net_psm_hotspot', 'office_gross', 'transit_gross', 'office_net', 'transit_net', 'rpc', 'a02fe', 'fe2rr', 'ao2rr', 'g2n', 'cobrm%', 'cobra%', 'ocara%', 'expired%', 'quick_cobra%', 'quick_ocara%', 'haps%', 'gsr2net' , '% Bid', 'positive_bid%', 'negative_bid%', 'gross_psm_gps%', 'gross_psm_fav_recent_history%', 'gross_psm_google_search%', 'gross_psm_stickylocation%', 'gross_psm_hotspot%', 'net_psm_gps%', 'net_psm_fav_recent_history%', 'net_psm_google_search%', 'net_psm_stickylocation%', 'net_psm_hotspot%', 'office_gross%', 'transit_gross%', 'office_net%', 'transit_net%', '% Gross customers', '% Gross Orders', 'ao2fe_delta_mean', 'ao2fe_delta_p25', 'ao2fe_delta_p50', 'ao2fe_delta_p75', 'ao2fe_delta_p90', 'ao2fe_delta_p95', 'fe2rr_delta_mean', 'fe2rr_delta_p25', 'fe2rr_delta_p50', 'fe2rr_delta_p75', 'fe2rr_delta_p90', 'fe2rr_delta_p95', 'ao2rr_delta_mean', 'ao2rr_delta_p25', 'ao2rr_delta_p50', 'ao2rr_delta_p75', 'ao2rr_delta_p90', 'ao2rr_delta_p95', 'aos_per_session_mean', 'aos_per_session_p25', 'aos_per_session_p50', 'aos_per_session_p75', 'aos_per_session_p90', 'aos_per_session_p95', 'aos_per_day_mean', 'aos_per_day_p25', 'aos_per_day_p50', 'aos_per_day_p75', 'aos_per_day_p90', 'aos_per_day_p95', 'fes_per_session_mean', 'fes_per_session_p25', 'fes_per_session_p50', 'fes_per_session_p75', 'fes_per_session_p90', 'fes_per_session_p95', 'fes_per_day_mean', 'fes_per_day_p25', 'fes_per_day_p50', 'fes_per_day_p75', 'fes_per_day_p90', 'fes_per_day_p95', 'rrs_per_session_mean', 'rrs_per_session_p25', 'rrs_per_session_p50', 'rrs_per_session_p75', 'rrs_per_session_p90', 'rrs_per_session_p95', 'rrs_per_day_mean', 'rrs_per_day_p25', 'rrs_per_day_p50', 'rrs_per_day_p75', 'rrs_per_day_p90', 'rrs_per_day_p95']

    cols2 = ['rider_canceltn_count','% rider_canceltn', 'tta_mean', 'tta_p25', 'tta_p50', 'tta_p75', 'tta_p90', 'tta_p95', 'average_rating', 'feedback_rating', '% rated_orders', 'rating_5', '% rating_5',
       'rating_4', '% rating_4', 'rating_3', '% rating_3', 'rating_2', '% rating_2', 'rating_1', '% rating_1',  'customer_call_captain',
       'captain_call_customer', 'customer_msg_captain', 'captain_msg_customer',
       'call_communication', '% call_communication', 'msg_communication', '% msg_communication', 'communication', '% communication', 'ocara_no',
       'ocara_with_comm', '% ocara_with_comm', 'ocara_without_comm','% ocara_without_comm',  'ocara_with_call', '% ocara_with_call', 
       'ocara_without_call', '% ocara_without_call', 'support_ticket', '% support_ticket']

    cols = cols1+cols2
    
    for col in cols:
        segment_agg_df[col] =  segment_agg_df[col].apply(lambda x: round(x,2))
    
    segment_agg_df = segment_agg_df[grouper+cols]
    segment_agg_df = segment_agg_df.drop(['aos_per_session_mean','aos_per_day_mean','fes_per_session_mean','fes_per_day_mean','rrs_per_session_mean','rrs_per_day_mean'],axis=1)
    
    return segment_agg_df.set_index(grouper)

In [30]:
%%time
segment_agg_df = agg_func(['segment'])

['segment']
CPU times: user 10.2 s, sys: 1.93 s, total: 12.2 s
Wall time: 13.1 s


In [31]:
segment_agg_df.shape

(8, 168)

In [32]:
segment_agg_df

,gross_customers,net_customers,ao_session,fe_session,rr_session,customer_rr_sessions,rr_count,net_count,cobrm_count,cobra_count,ocara_count,expired_count,aborted_count,canceltn_count,quick_cobra,quick_ocara,cobra_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,haps_orders,bid_orders,negative_bid,positive_bid,gross_psm_gps,gross_psm_fav_recent_history,gross_psm_google_search,gross_psm_stickylocation,gross_psm_hotspot,net_psm_gps,net_psm_fav_recent_history,net_psm_google_search,net_psm_stickylocation,net_psm_hotspot,office_gross,transit_gross,office_net,transit_net,rpc,a02fe,fe2rr,ao2rr,g2n,cobrm%,cobra%,ocara%,expired%,quick_cobra%,quick_ocara%,haps%,gsr2net,% Bid,positive_bid%,negative_bid%,gross_psm_gps%,gross_psm_fav_recent_history%,gross_psm_google_search%,gross_psm_stickylocation%,gross_psm_hotspot%,net_psm_gps%,net_psm_fav_recent_history%,net_psm_google_search%,net_psm_stickylocation%,net_psm_hotspot%,office_gross%,transit_gross%,office_net%,transit_net%,% Gross customers,% Gross Orders,ao2fe_delta_mean,ao2fe_delta_p25,ao2fe_delta_p50,ao2fe_delta_p75,ao2fe_delta_p90,ao2fe_delta_p95,fe2rr_delta_mean,fe2rr_delta_p25,fe2rr_delta_p50,fe2rr_delta_p75,fe2rr_delta_p90,fe2rr_delta_p95,ao2rr_delta_mean,ao2rr_delta_p25,ao2rr_delta_p50,ao2rr_delta_p75,ao2rr_delta_p90,ao2rr_delta_p95,aos_per_session_p25,aos_per_session_p50,aos_per_session_p75,aos_per_session_p90,aos_per_session_p95,aos_per_day_p25,aos_per_day_p50,aos_per_day_p75,aos_per_day_p90,aos_per_day_p95,fes_per_session_p25,fes_per_session_p50,fes_per_session_p75,fes_per_session_p90,fes_per_session_p95,fes_per_day_p25,fes_per_day_p50,fes_per_day_p75,fes_per_day_p90,fes_per_day_p95,rrs_per_session_p25,rrs_per_session_p50,rrs_per_session_p75,rrs_per_session_p90,rrs_per_session_p95,rrs_per_day_p25,rrs_per_day_p50,rrs_per_day_p75,rrs_per_day_p90,rrs_per_day_p95,rider_canceltn_count,% rider_canceltn,tta_mean,tta_p25,tta_p50,tta_p75,tta_p90,tta_p95,average_rating,feedback_rating,% rated_orders,rating_5,% rating_5,rating_4,% rating_4,rating_3,% rating_3,rating_2,% rating_2,rating_1,% rating_1,customer_call_captain,captain_call_customer,customer_msg_captain,captain_msg_customer,call_communication,% call_communication,msg_communication,% msg_communication,communication,% communication,ocara_no,ocara_with_comm,% ocara_with_comm,ocara_without_comm,% ocara_without_comm,ocara_with_call,% ocara_with_call,ocara_without_call,% ocara_without_call,support_ticket,% support_ticket
segment,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1-HH-Active,434017,379640,1467504.0,1119569.0,547167.0,767072.0,905093.0,548258.0,11658.0,113689.0,165917.0,64200.0,1362.0,291264.0,14749.0,14999.0,118.64,59.00,102.33,161.00,227.50,271.00,525.97,162.00,350.00,658.00,1133.0,1595.07,375625.0,101239.0,28431.0,72808.0,518579.0,15174.0,52722.0,110951.0,54364.0,316894.0,7623.0,30005.0,68703.0,32523.0,133296.0,123730.0,70284.0,69353.0,1.44,76.29,48.87,37.29,60.57,1.29,12.56,18.33,7.24,1.63,1.66,68.51,71.47,11.19,71.92,28.08,57.30,1.68,5.83,12.26,6.01,57.80,1.39,5.47,12.53,5.93,14.73,13.67,12.82,12.65,100.0,100.0,52.69,15.50,29.75,57.67,111.00,171.00,127.96,26.00,50.00,125.00,318.00,524.00,173.24,51.00,92.00,191.00,397.88,606.00,1.0,1.00,1.33,1.71,2.00,1.00,1.50,2.00,3.00,4.00,1.00,1.50,2.00,2.50,3.00,1.50,2.25,3.50,5.00,7.00,1.00,1.00,1.00,1.67,2.00,1.00,1.00,2.00,2.33,3.00,134149.0,14.82,84.87,6.67,20.50,80.00,201.00,335.00,4.55,104614.0,19.08,82286.0,78.66,11203.0,10.71,3623.0,3.46,1600.0,1.53,5902.0,5.64,4757.0,369067.0,204695.0,373323.0,371342.0,41.03,427121.0,47.19,554264.0,61.24,165917.0,109381.0,65.93,56536.0,34.07,65723.0,39.61,100194.0,60.39,5679.0,1.04
2-HH-inActive,17332,14989,552406.0,269578.0,10816.0,14832.0,32734.0,20286.0,484.0,4258.0,5731.0,1935.0,40.0,10473.0,638.0,589.0,11

In [34]:
segment_agg_df.reset_index().to_clipboard(index=False)

In [35]:
segment_os = agg_func(['segment','os'])
segment_os

['segment', 'os']


gross_customers  net_customers  ao_session  fe_session  \
segment       os                                                                
1-HH-Active   Android           382876         336021   1330282.0   1016969.0   
              iOS                51141          43619    137222.0    102600.0   
2-HH-inActive Android            17127          14817    503822.0    249692.0   
              iOS                  205            172     48584.0     19886.0   
3-DAILY       Android            73655          73564   1557746.0   1389492.0   
              iOS                34740          34728    620925.0    522683.0   
4-WEEKLY      Android           212620         211358   2658399.0   2293592.0   
              iOS                83564          83058    908945.0    736314.0   
5-MONTHLY     Android          1226544        1130893   7634970.0   6179372.0   
              iOS               342527         313048   1965863.0   1442041.0   
6-QUARTERLY   Android           397333         388578   2074089.0   1631624.0   
              iOS                82480          81211    429224.0    304733.0   
7-RR-ACTIVE   Android            25564             83    112067.0     74738.0   
              iOS                 7457              9     33445.0     17173.0   
8-RR-INACTIVE Android             1860              5    174496.0    108083.0   
              iOS                   73              0     37124.0     16805.0   

                       rr_session  customer_rr_sessions   rr_count  net_count  \
segment       os                                                                
1-HH-Active   Android    487255.0              670296.0   795307.0   483646.0   
              iOS         59912.0               96776.0   109786.0    64612.0   
2-HH-inActive Android      9622.0               13027.0    32385.0    20084.0   
              iOS          1194.0                1805.0      349.0      202.0   
3-DAILY       Android   1138053.0             2325284.0  3035890.0  1914360.0   
              iOS        437692.0             1204277.0  1491711.0   919867.0   
4-WEEKLY      Android   1669833.0             2555653.0  3238352.0  2001640.0   
              iOS        561287.0             1127427.0  1363053.0   818054.0   
5-MONTHLY     Android   3666360.0             5160015.0  6375401.0  4019198.0   
              iOS        964643.0             1698908.0  2001490.0  1222314.0   
6-QUARTERLY   Android    885144.0             1352979.0  1962409.0  1308076.0   
              iOS        189987.0              362145.0   426839.0   279482.0   
7-RR-ACTIVE   Android     28117.0               34357.0    40576.0       85.0   
              iOS          8298.0               10562.0    12175.0        9.0   
8-RR-INACTIVE Android      4295.0                5500.0     2932.0        6.0   
              iOS           952.0                1455.0      105.0        0.0   

                       cobrm_count  cobra_count  ocara_count  expired_count  \
segment       os                                                              
1-HH-Active   Android      10714.0      97906.0     146183.0        55613.0   
              iOS            944.0      15783.0      19734.0         8587.0   
2-HH-inActive Android        479.0       4200.0       5665.0         1917.0   
              iOS              5.0         58.0         66.0           18.0   
3-DAILY       Android      20124.0     468173.0     481879.0       149891.0   
              iOS           7843.0     200604.0     276038.0        86823.0   
4-WEEKLY      Android      25351.0     512027.0     521869.0       175549.0   
              iOS           8377.0     197273.0     248752.0        90030.0   
5-MONTHLY     Android      56191.0     978131.0     983616.0       333033.0   
              iOS          13634.0     300343.0     339024.0       125004.0   
6-QUARTERLY   Android      17692.0     269335.0     267232.0        98470.0   
              iOS           3159.0      58840.0      61078.0        24034.0   

In [36]:
segment_os.reset_index().to_clipboard(index=False)

In [37]:
segment_travel = agg_func(['segment','travelridehailing'])
segment_travel

['segment', 'travelridehailing']


gross_customers  net_customers  ao_session  \
segment       travelridehailing                                               
1-HH-Active   blank                        49292          42209    131726.0   
              no                          166935         150727    600139.0   
              yes                         217790         186704    735639.0   
2-HH-inActive blank                          216            186     58297.0   
              no                            6891           6171    203218.0   
              yes                          10225           8632    290891.0   
3-DAILY       blank                        29919          29910    537009.0   
              no                           10503          10473    212546.0   
              yes                          67973          67909   1429116.0   
4-WEEKLY      blank                        71993          71546    785223.0   
              no                           37050          36861    447917.0   
              yes                         187141         186009   2334204.0   
5-MONTHLY     blank                       298580         272674   1718658.0   
              no                          318817         298891   1989206.0   
              yes                         951674         872376   5892969.0   
6-QUARTERLY   blank                        74801          73656    394558.0   
              no                          114723         112975    638993.0   
              yes                         290289         283158   1469762.0   
7-RR-ACTIVE   blank                         6792              8     30264.0   
              no                            5700             28     26372.0   
              yes                          20529             56     88876.0   
8-RR-INACTIVE blank                           82              0     34255.0   
              no                             347              2     47850.0   
              yes                           1504              3    129515.0   

                                 fe_session  rr_session  customer_rr_sessions  \
segment       travelridehailing                                                 
1-HH-Active   blank                 98631.0     58164.0               93452.0   
              no                   450079.0    204598.0              289067.0   
              yes                  570859.0    284405.0              384553.0   
2-HH-inActive blank                 19213.0      1125.0                1700.0   
              no                    97347.0      3061.0                4313.0   
              yes                  153018.0      6630.0                8819.0   
3-DAILY       blank                451375.0    378944.0             1037362.0   
              no                   187474.0    152997.0              323184.0   
              yes                 1273326.0   1043804.0             2169015.0   
4-WEEKLY      blank                634208.0    486352.0              974672.0   
              no                   377132.0    269325.0              417084.0   
              yes                 2018566.0   1475443.0             2291324.0   
5-MONTHLY     blank               1251089.0    843183.0             1479146.0   
              no                  1555527.0    879671.0             1259466.0   
              yes                 4814797.0   2908149.0             4120311.0   
6-QUARTERLY   blank                278518.0    177129.0              332645.0   
              no                   490053.0    251859.0              391440.0   
              yes                 1167786.0    646143.0              991039.0   
7-RR-ACTIVE   blank                 15449.0      7658.0                9567.0   
              no                    16552.0      5744.0                7608.0   
              yes                   59910.0     23013.0               27744.0   
8-RR-INACTIVE blank                 15371.0       886.0                1366.0   
              no                  

In [38]:
segment_travel.reset_index().to_clipboard(index=False)

In [39]:
segment_qc = agg_func(['segment','quick_commerce'])
segment_qc

['segment', 'quick_commerce']


gross_customers  net_customers  ao_session  \
segment       quick_commerce                                               
1-HH-Active   blank                     49292          42209    131726.0   
              no                       165806         148888    605294.0   
              yes                      218919         188543    730484.0   
2-HH-inActive blank                       216            186     58297.0   
              no                         6873           6078    200712.0   
              yes                       10243           8725    293397.0   
3-DAILY       blank                     29919          29910    537009.0   
              no                        10350          10311    232290.0   
              yes                       68126          68071   1409372.0   
4-WEEKLY      blank                     71993          71546    785223.0   
              no                        33723          33456    437110.0   
              yes                      190468         189414   2345011.0   
5-MONTHLY     blank                    298580         272674   1718658.0   
              no                       280624         260472   1837627.0   
              yes                      989867         910795   6044548.0   
6-QUARTERLY   blank                     74801          73656    394558.0   
              no                       106539         104781    616633.0   
              yes                      298473         291352   1492122.0   
7-RR-ACTIVE   blank                      6792              8     30264.0   
              no                         6120             23     28721.0   
              yes                       20109             61     86527.0   
8-RR-INACTIVE blank                        82              0     34255.0   
              no                          388              1     49729.0   
              yes                        1463              4    127636.0   

                              fe_session  rr_session  customer_rr_sessions  \
segment       quick_commerce                                                 
1-HH-Active   blank              98631.0     58164.0               93452.0   
              no                457949.0    208705.0              290483.0   
              yes               562989.0    280298.0              383137.0   
2-HH-inActive blank              19213.0      1125.0                1700.0   
              no                 98516.0      3071.0                4257.0   
              yes               151849.0      6620.0                8875.0   
3-DAILY       blank             451375.0    378944.0             1037362.0   
              no                204154.0    160435.0              321925.0   
              yes              1256646.0   1036366.0             2170274.0   
4-WEEKLY      blank             634208.0    486352.0              974672.0   
              no                368793.0    253411.0              383376.0   
              yes              2026905.0   1491357.0             2325032.0   
5-MONTHLY     blank            1251089.0    843183.0             1479146.0   
              no               1448212.0    798337.0             1129975.0   
              yes              4922112.0   2989483.0             4249802.0   
6-QUARTERLY   blank             278518.0    177129.0              332645.0   
              no                476609.0    243496.0              374875.0   
              yes              1181230.0    654506.0             1007604.0   
7-RR-ACTIVE   blank              15449.0      7658.0                9567.0   
              no                 18541.0      6199.0                8048.0   
              yes                57921.0     22558.0               27304.0   
8-RR-INACTIVE blank              15371.0       886.0                1366.0   
              no                 29342.0       828.0                1106.0   
              yes                80175.0      3533.0                4483.0   

                        

In [40]:
segment_qc.reset_index().to_clipboard(index=False)

In [43]:
segment_travel_qc = agg_func(['segment','travelridehailing','quick_commerce'])
segment_travel_qc

['segment', 'travelridehailing', 'quick_commerce']


gross_customers  \
segment       travelridehailing quick_commerce                    
1-HH-Active   blank             blank                     49292   
              no                no                       105241   
                                yes                       61694   
              yes               no                        60565   
                                yes                      157225   
2-HH-inActive blank             blank                       216   
              no                no                         4150   
                                yes                        2741   
              yes               no                         2723   
                                yes                        7502   
3-DAILY       blank             blank                     29919   
              no                no                         4486   
                                yes                        6017   
              yes               no                         5864   
                                yes                       62109   
4-WEEKLY      blank             blank                     71993   
              no                no                        15586   
                                yes                       21464   
              yes               no                        18137   
                                yes                      169004   
5-MONTHLY     blank             blank                    298580   
              no                no                       152199   
                                yes                      166618   
              yes               no                       128425   
                                yes                      823249   
6-QUARTERLY   blank             blank                     74801   
              no                no                        59514   
                                yes                       55209   
              yes               no                        47025   
                                yes                      243264   
7-RR-ACTIVE   blank             blank                      6792   
              no                no                         2984   
                                yes                        2716   
              yes               no                         3136   
                                yes                       17393   
8-RR-INACTIVE blank             blank                        82   
              no                no                          182   
                                yes                         165   
              yes               no                          206   
                                yes                        1298   

                                                net_customers  ao_session  \
segment       travelridehailing quick_commerce                              
1-HH-Active   blank             blank                   42209    131726.0   
              no                no                      95677    384032.0   
                                yes                     55050    216107.0   
              yes               no                      53211    221262.0   
                                yes                    133493    514377.0   
2-HH-inActive blank             blank                     186     58297.0   
              no                no                       3743    118519.0   
                                yes                      2428     84699.0   
              yes               no                       2335     82193.0   
                                yes                      6297    208698.0   
3-DAILY       blank             blank                   29910    537009.0   
              no                no                       4465     94190.0   
                                yes                      6008    118356.0   
              yes               no                       5846    138100.0   
 

In [44]:
segment_travel_qc.reset_index().to_clipboard(index=False)

In [41]:
segment_office = agg_func(['segment','office_tag'])
segment_office

['segment', 'office_tag']


gross_customers  net_customers  ao_session  \
segment       office_tag                                               
1-HH-Active   office                69231          64987    253508.0   
              other                364786         314653   1213996.0   
2-HH-inActive office                 2494           2318     43597.0   
              other                 14838          12671    508809.0   
3-DAILY       office                87009          86960   1759595.0   
              other                 21386          21332    419076.0   
4-WEEKLY      office               208956         208024   2559200.0   
              other                 87228          86392   1008144.0   
5-MONTHLY     office               683412         650168   4620346.0   
              other                885659         793773   4980487.0   
6-QUARTERLY   office               135120         133170    827961.0   
              other                344693         336619   1675352.0   
7-RR-ACTIVE   office                   44              3       401.0   
              other                 32977             89    145111.0   
8-RR-INACTIVE office                    2              0       561.0   
              other                  1931              5    211059.0   

                          fe_session  rr_session  customer_rr_sessions  \
segment       office_tag                                                 
1-HH-Active   office        198858.0     98665.0              139722.0   
              other         920711.0    448502.0              627350.0   
2-HH-inActive office         21575.0      1253.0                1703.0   
              other         248003.0      9563.0               13129.0   
3-DAILY       office       1541566.0   1271510.0             2905782.0   
              other         370609.0    304235.0              623779.0   
4-WEEKLY      office       2178765.0   1614173.0             2734068.0   
              other         851141.0    616947.0              949012.0   
5-MONTHLY     office       3757750.0   2385104.0             3716502.0   
              other        3863663.0   2245899.0             3142421.0   
6-QUARTERLY   office        676611.0    412310.0              708743.0   
              other        1259746.0    662821.0             1006381.0   
7-RR-ACTIVE   office           253.0       140.0                 192.0   
              other          91658.0     36275.0               44727.0   
8-RR-INACTIVE office           415.0       232.0                 329.0   
              other         124473.0      5015.0                6626.0   

                           rr_count  net_count  cobrm_count  cobra_count  \
segment       office_tag                                                   
1-HH-Active   office       164066.0   104712.0       1752.0      20207.0   
              other        741027.0   443546.0       9906.0      93482.0   
2-HH-inActive office         5189.0     3491.0         73.0        619.0   
              other         27545.0    16795.0        411.0       3639.0   
3-DAILY       office      3735070.0  2327465.0      23222.0     556075.0   
              other        792531.0   506762.0       4745.0     112702.0   
4-WEEKLY      office      3431082.0  2092405.0      25486.0     536262.0   
              other       1170323.0   727289.0       8242.0     173038.0   
5-MONTHLY     office      4602329.0  2879487.0      38090.0     718442.0   
              other       3774562.0  2362025.0      31735.0     560032.0   
6-QUARTERLY   office      1045888.0   681464.0       9633.0     154069.0   
              other       1343360.0   906094.0      11218.0     174106.0   
7-RR-ACTIVE   office           68.0        3.0          2.0         23.0   
              other         52683.0       91.0       1258.0      18576.0   
8-RR-INACTIVE office            5.0        0.0          0.0          5.0   
              other          3032.0        6.0         74.0       1101.0   

                          o

In [42]:
segment_office.reset_index().to_clipboard(index=False)

In [45]:
segment_office_travel_qc = agg_func(['segment','office_tag','travelridehailing','quick_commerce'])
segment_office_travel_qc

['segment', 'office_tag', 'travelridehailing', 'quick_commerce']


gross_customers  \
segment       office_tag travelridehailing quick_commerce                    
1-HH-Active   office     blank             blank                     10507   
                         no                no                        14590   
                                           yes                        8845   
                         yes               no                         8694   
                                           yes                       26595   
...                                                                    ...   
8-RR-INACTIVE other      blank             blank                        81   
                         no                no                          182   
                                           yes                         165   
                         yes               no                          206   
                                           yes                        1297   

                                                           net_customers  \
segment       office_tag travelridehailing quick_commerce                  
1-HH-Active   office     blank             blank                    9794   
                         no                no                      13910   
                                           yes                      8379   
                         yes               no                       8214   
                                           yes                     24690   
...                                                                  ...   
8-RR-INACTIVE other      blank             blank                       0   
                         no                no                          1   
                                           yes                         1   
                         yes               no                          0   
                                           yes                         3   

                                                           ao_session  \
segment       office_tag travelridehailing quick_commerce               
1-HH-Active   office     blank             blank              30529.0   
                         no                no                 57926.0   
                                           yes                34288.0   
                         yes               no                 34484.0   
                                           yes                96281.0   
...                                                               ...   
8-RR-INACTIVE other      blank             blank              34160.0   
                         no                no                 24727.0   
                                           yes                23051.0   
                         yes               no                 24852.0   
                                           yes               104269.0   

                                                           fe_session  \
segment       office_tag travelridehailing quick_commerce               
1-HH-Active   office     blank             blank              24106.0   
                         no                no                 44261.0   
                                           yes                26372.0   
                         yes               no                 27112.0   
                                           yes                77007.0   
...                                                               ...   
8-RR-INACTIVE other      blank             blank              15298.0   
                         no                no                 14372.0   
                                           yes                13190.0   
                         yes               no                 14864.0   
                                           yes                66749.0   

                                                           rr_session  \
segment       office_tag travelridehailing quick_commerce               
1-HH-Acti

In [46]:
segment_office_travel_qc.reset_index().to_clipboard(index=False)